# (2026-08-06) Implementing vLLM inference for data gen and RL training.
This notebook is an offline scratchpad for optimizing the inference speed of graph-augmented LLMs. Specificaly,
the RPE mask implementation.

## Todos
1. Fetch a small model compatible with vLLM and run inference. Verify it's faster.
2. Identify parameters to tune for inf performance (batch size, seq len etc).
3. Modify attention mask in the same way as GREP/GEGR. (Learnable mask)
4. Integrate this accelerated inference impl. works with RL.
    4.1 Can we have an RL baseline to verify this implementation works? 

## Later
- Bring in innovations from WIRE, WEAVE and other GNN-LLM projects. Particularly:
    - GNN+RoPE for relative information in both text and graph.
    - Centroid bias

In [1]:
import time

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
MAX_NEW_TOKENS = 64


## Load small model


In [2]:
# HF transformers baseline (CPU, fp32) — reference point for todo #1.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
print(f"device={model.device}  dtype={model.dtype}")

t0 = time.perf_counter()
hf_texts = []
for p in prompts:
    inputs = tokenizer(p, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    hf_texts.append(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
hf_seconds = time.perf_counter() - t0

for p, t in zip(prompts, hf_texts):
    print(f"{p!r} -> {t!r}")
print(f"\nHF baseline: {hf_seconds:.1f}s for {len(prompts)} prompts x {MAX_NEW_TOKENS} new tokens")


/Users/jporras/.venv-vllm-metal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:01<06:47,  1.41s/it]

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 204.92it/s]

device=cpu  dtype=torch.float32


'Hello, my name is' -> ' Alex and I am a 17 year old male. I have been diagnosed with a rare genetic disorder called X-linked recessive. My mother has the same condition as me. What can I do to help my brother understand that he will not be affected by this disease? Is there anything I can say or do to'
'The president of the United States is' -> ' a very important person. He or she has many important jobs to do every day. The president is like the boss of the country. He or she makes decisions for the whole country. The president also helps make sure that everyone in the country gets along with each other. The president is usually elected by people who vote for'
'The capital of France is' -> ' Paris. It was founded in 789 AD by Charlemagne, the last king of the Carolingian dynasty. The city has a long and rich history dating back to the Roman Empire. In fact, it was the first European capital to be built on land that would later become the French Riviera.\n'
'The future of AI is' -> ' 

## vLLM setup

In [3]:
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=MAX_NEW_TOKENS)

llm = LLM(model=MODEL_ID, max_model_len=2048)

t0 = time.perf_counter()
outputs = llm.generate(prompts, sampling_params)
vllm_seconds = time.perf_counter() - t0

for o in outputs:
    print(f"{o.prompt!r} -> {o.outputs[0].text!r}")


INFO 08-07 17:48:44 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 08-07 17:48:44 [__init__.py:54] - metal -> vllm_metal:register


INFO 08-07 17:48:44 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 08-07 17:48:45 [__init__.py:237] Platform plugin metal is activated


INFO 08-07 17:48:48 [importing.py:88] Triton not installed or not compatible; certain GPU-related functions will not be available.


...

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:  25%|██▌       | 1/4 [00:06<00:18,  6.10s/it, est. speed input: 0.82 toks/s, output: 10.49 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:06<00:00,  6.10s/it, est. speed input: 3.60 toks/s, output: 41.88 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:06<00:00,  1.53s/it, est. speed input: 3.60 toks/s, output: 41.88 toks/s]

'Hello, my name is' -> ' Emily and I am a PhD student in Computer Science and Mathematics at the University of California, Berkeley. Before coming to UC Berkeley, I graduated with a BSc from the University of East Anglia and an MSc from the University of Oxford. I started my PhD research at UC Berkeley in Fall 2020'
'The president of the United States is' -> ' a very important person. This person is called the President of the United States. The president of the United States is in charge of the government of the United States. This person is also called the president. Most people in the United States think the President of the United States is the best man in the world. But many'
'The capital of France is' -> ' Paris.\nA. True\nB. False\nAnswer:\n\nA\n\nWhat does the phrase "strange meetings" in the first paragraph refer to?\nA. Meetings that occurred in strange places\nB. Meetings that occurred in unusual circumstances\nC. Meetings that occurred at different times\nD. Meetings that o

In [4]:
vllm_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
print(f"HF transformers (cpu, sequential): {hf_seconds:6.1f}s  ({len(prompts) * MAX_NEW_TOKENS / hf_seconds:6.1f} tok/s)")
print(f"vLLM metal (batched):              {vllm_seconds:6.1f}s  ({vllm_tokens / vllm_seconds:6.1f} tok/s)")
print(f"speedup: {hf_seconds / vllm_seconds:.1f}x")


HF transformers (cpu, sequential):    5.5s  (  46.7 tok/s)
vLLM metal (batched):                 6.3s  (  40.8 tok/s)
speedup: 0.9x
